# 01. Data Compilation and Loading

This notebook covers the first step in a typical QSAR workflow:
- Loading a dataset with chemical structures (SMILES) and biological activities
- Performing initial data quality checks
- Exploratory data analysis
- Ensuring the dataset is representative of the chemical space of interest

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors

# Set style for plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1.1 Load the Dataset

Load the dataset containing chemical structures (SMILES) and their associated biological activities.

In [ ]:
# Load the example dataset
data_path = '../Data/testcase.csv'
df = pd.read_csv(data_path)

print(f"Dataset loaded successfully!")
print(f"Number of compounds: {len(df)}")
print(f"\nFirst few rows:")
df.head()

## 1.2 Data Quality Checks

Perform initial quality checks on the dataset.

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Check for duplicates
duplicates = df.duplicated(subset=['Smiles']).sum()
print(f"Number of duplicate SMILES: {duplicates}")

if duplicates > 0:
    print("\nDuplicate compounds:")
    print(df[df.duplicated(subset=['Smiles'], keep=False)].sort_values('Smiles'))

In [ ]:
# Validate SMILES strings
def is_valid_smiles(smiles):
    """Check if a SMILES string is valid."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol is not None
    except:
        return False

df['valid_smiles'] = df['Smiles'].apply(is_valid_smiles)
invalid_count = (~df['valid_smiles']).sum()

print(f"Valid SMILES: {df['valid_smiles'].sum()}")
print(f"Invalid SMILES: {invalid_count}")

if invalid_count > 0:
    print("\nInvalid SMILES:")
    print(df[~df['valid_smiles']][['Smiles', 'pChEMBL']])

## 1.3 Exploratory Data Analysis

Analyze the distribution of biological activities and basic molecular properties.

In [ ]:
# Statistical summary of activity values
print("Activity Statistics (pChEMBL):")
print(df['pChEMBL'].describe())

In [ ]:
# Plot activity distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(df['pChEMBL'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('pChEMBL Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Biological Activity')
axes[0].axvline(df['pChEMBL'].mean(), color='red', linestyle='--', label='Mean')
axes[0].axvline(df['pChEMBL'].median(), color='green', linestyle='--', label='Median')
axes[0].legend()

# Box plot
axes[1].boxplot(df['pChEMBL'])
axes[1].set_ylabel('pChEMBL Value')
axes[1].set_title('Box Plot of Biological Activity')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate basic molecular properties
def calculate_basic_properties(smiles):
    """Calculate basic molecular properties from SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None, None, None
    
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    
    return mw, logp, hbd, hba

# Calculate properties for valid SMILES
valid_df = df[df['valid_smiles']].copy()
properties = valid_df['Smiles'].apply(calculate_basic_properties)
valid_df[['MolWt', 'LogP', 'HBD', 'HBA']] = pd.DataFrame(properties.tolist(), index=valid_df.index)

print("\nMolecular Properties Statistics:")
print(valid_df[['MolWt', 'LogP', 'HBD', 'HBA']].describe())

In [ ]:
# Plot molecular properties distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(valid_df['MolWt'], bins=20, edgecolor='black', alpha=0.7)
axes[0, 0].set_xlabel('Molecular Weight (Da)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Molecular Weight Distribution')

axes[0, 1].hist(valid_df['LogP'], bins=20, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('LogP')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('LogP Distribution')

axes[1, 0].hist(valid_df['HBD'], bins=range(int(valid_df['HBD'].max())+2), edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('H-Bond Donors')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('H-Bond Donors Distribution')

axes[1, 1].hist(valid_df['HBA'], bins=range(int(valid_df['HBA'].max())+2), edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('H-Bond Acceptors')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('H-Bond Acceptors Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation between molecular properties and activity
correlation_data = valid_df[['pChEMBL', 'MolWt', 'LogP', 'HBD', 'HBA']]
correlation_matrix = correlation_data.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Activity vs Molecular Properties')
plt.tight_layout()
plt.show()

## 1.4 Summary

Summarize the data compilation and quality assessment.

In [ ]:
print("="*60)
print("DATA COMPILATION SUMMARY")
print("="*60)
print(f"Total compounds: {len(df)}")
print(f"Valid SMILES: {df['valid_smiles'].sum()}")
print(f"Invalid SMILES: {invalid_count}")
print(f"Duplicate SMILES: {duplicates}")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nActivity range: {df['pChEMBL'].min():.2f} - {df['pChEMBL'].max():.2f}")
print(f"Activity mean ± std: {df['pChEMBL'].mean():.2f} ± {df['pChEMBL'].std():.2f}")
print("\nDataset is ready for descriptor calculation!")
print("="*60)

## Next Steps

The data has been compiled and validated. The next notebook (02_descriptor_calculation.ipynb) will:
- Standardize SMILES strings
- Calculate diverse molecular descriptors and fingerprints
- Save the featurized dataset for further analysis